In [ ]:
!pip install -q cornac

In [ ]:
import pandas as pd, numpy as np, os

train = pd.read_parquet("/kaggle/input/notebooks/fatimawaseem1111/parquet/train.parquet")
test = pd.read_parquet("/kaggle/input/notebooks/fatimawaseem1111/parquet/test.parquet")
ground_truth = test.groupby("customer_id")["article_id"].apply(set).to_dict()

for root, dirs, files in os.walk("/kaggle/input/notebooks"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
from cornac.data import Dataset, ImageModality
from cornac.models import VBPR

uir = train[["customer_id", "article_id"]].drop_duplicates()
uir["rating"] = 1.0
train_set = Dataset.from_uir(uir.itertuples(index=False, name=None), seed=42)
item_ids = np.array(list(train_set.iid_map.keys()))

FEAT_DIR = None
for root, dirs, files in os.walk("/kaggle/input/notebooks"):
    if "item_list.csv" in files:
        FEAT_DIR = root + "/"
print("Features folder:", FEAT_DIR)
feats = {
    "ResNet (image)": np.load(FEAT_DIR + "img_emb.npy"),
    "SBERT (text)": np.load(FEAT_DIR + "text_emb.npy"),
    "FashionCLIP (image)": np.load(FEAT_DIR + "fclip_img.npy"),
    "FashionCLIP (text)": np.load(FEAT_DIR + "fclip_txt.npy"),
}
feats["ResNet + SBERT"] = np.hstack([feats["ResNet (image)"], feats["SBERT (text)"]])
feats["FashionCLIP (image + text)"] = np.hstack([feats["FashionCLIP (image)"], feats["FashionCLIP (text)"]])

In [ ]:
print("Features folder:", FEAT_DIR)

item_list = pd.read_csv(FEAT_DIR + "item_list.csv").iloc[:, 0].astype(str).str.zfill(10).tolist()
print(len(item_list), "items | first:", item_list[0])

feats = {
    "ResNet (image)": np.load(FEAT_DIR + "img_emb.npy"),
    "SBERT (text)": np.load(FEAT_DIR + "text_emb.npy"),
    "FashionCLIP (image)": np.load(FEAT_DIR + "fclip_img.npy"),
    "FashionCLIP (text)": np.load(FEAT_DIR + "fclip_txt.npy"),
}
feats["ResNet + SBERT"] = np.hstack([feats["ResNet (image)"], feats["SBERT (text)"]])
feats["FashionCLIP (image + text)"] = np.hstack([feats["FashionCLIP (image)"], feats["FashionCLIP (text)"]])

for name, f in feats.items():
    print(name, f.shape)

In [ ]:
def run_vbpr(feat_matrix, k=64, k2=64, epochs=20, lr=0.005, seed=42):
    modality = ImageModality(features=feat_matrix, ids=item_list, normalized=True)
    train_set.add_modalities(item_image=modality)
    model = VBPR(k=k, k2=k2, n_epochs=epochs, batch_size=1000, learning_rate=lr,
                 lambda_w=0.01, lambda_b=0.01, lambda_e=0.0, use_gpu=True, seed=seed, verbose=True)
    model.fit(train_set)
    recs = {c: item_ids[np.argsort(-model.score(train_set.uid_map[c]))[:12]].tolist()
            for c in ground_truth}
    return evaluate(recs, ground_truth)

In [ ]:
def apk(actual, predicted, k=12):
    if not actual:
        return 0.0
    score = 0.0
    hits = 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def recall_at_k(actual, predicted, k=12):
    if not actual:
        return 0.0
    hits = len(set(predicted[:k]) & actual)
    return hits / len(actual)

def evaluate(recs, ground_truth, k=12):
    maps, recalls = [], []
    for cust, actual in ground_truth.items():
        predicted = recs.get(cust, [])
        maps.append(apk(actual, predicted, k))
        recalls.append(recall_at_k(actual, predicted, k))
    return {"MAP@12": np.mean(maps), "Recall@12": np.mean(recalls)}

print(apk({"A", "B"}, ["X", "A", "Y", "B"]))  # check: should be 0.5

In [ ]:
vbpr_results = {}
for name, f in feats.items():
    print("Training:", name, flush=True)
    vbpr_results[name] = run_vbpr(f)
    print(name, vbpr_results[name], flush=True)

vbpr_df = pd.DataFrame(vbpr_results).T.sort_values("MAP@12", ascending=False)
print(vbpr_df)
vbpr_df.to_csv("/kaggle/working/vbpr_results.csv")

In [ ]:
    return evaluate(recs, ground_truth), recs

In [ ]:
seed_rows = []
for seed in [7, 123]:
    for name, f in feats.items():
        res, _ = run_vbpr(f, seed=seed)
        seed_rows.append({"model": name, "seed": seed, **res})
        print(seed, name, res, flush=True)

for name, res in vbpr_results.items():
    seed_rows.append({"model": name, "seed": 42, **res})

seed_df = pd.DataFrame(seed_rows)
summary = seed_df.groupby("model")[["MAP@12", "Recall@12"]].agg(["mean", "std"])
summary = summary.sort_values(("MAP@12", "mean"), ascending=False)
print(summary)
summary.to_csv("/kaggle/working/vbpr_seeds_summary.csv")

In [ ]:
recent = (train.sort_values("t_dat", ascending=False)
               .groupby("customer_id")["article_id"]
               .apply(lambda x: list(dict.fromkeys(x))[:12]))

res_best, recs_best = run_vbpr(feats["FashionCLIP (text)"], seed=42)

recs_hybrid = {}
for cust in ground_truth:
    items = recent.get(cust, [])
    items = items + [i for i in recs_best[cust] if i not in items]
    recs_hybrid[cust] = items[:12]

print("Repurchase + VBPR (FashionCLIP text):", evaluate(recs_hybrid, ground_truth))